In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import glob
from pathlib import Path
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
from src.text_encoder import encode_text, decode_prediction, char_to_num

I0000 00:00:1779797247.580119  166145 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779797260.881432  166145 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1765 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


SETUP

In [2]:
PROJECT_ROOT = "/home/hasan/coding/MoneyLens/ai"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
 
from src.ocr_config import *
from src.ocr_model import build_ocr_model, CTCLayer, build_inference_model, ResidualCNNBlock
from src.text_encoder import encode_text, decode_prediction

Custom Callback

In [3]:
class OCRTrainingLogger(keras.callbacks.Callback):
    """
    Custom callback yang:
    1. Print ringkasan per epoch ke console dengan format rapi
    2. Simpan history ke CSV setiap epoch (fault-tolerant)
    3. Deteksi plateau val_loss dan beri warning dini
    4. Catat waktu training per epoch
    """
 
    def __init__(self, csv_path: str, patience_warn: int = 5):
        super().__init__()
        self.csv_path     = csv_path
        self.patience_warn= patience_warn
        self.history_rows = []
        self.best_val     = np.inf
        self.plateau_count= 0
        self._epoch_start = None
 
    def on_epoch_begin(self, epoch, logs=None):
        import time
        self._epoch_start = time.time()
 
    def on_epoch_end(self, epoch, logs=None):
        import time
        elapsed = time.time() - self._epoch_start if self._epoch_start else 0
        logs    = logs or {}
 
        val_loss  = logs.get("val_loss", np.nan)
        train_loss= logs.get("loss", np.nan)
        lr        = float(self.model.optimizer.learning_rate)
 
        # Plateau detection
        if val_loss < self.best_val:
            self.best_val   = val_loss
            self.plateau_count = 0
            flag = "✅"
        else:
            self.plateau_count += 1
            flag = f"⏸ ({self.plateau_count})" if self.plateau_count < self.patience_warn else "⚠️ PLATEAU"
 
        print(f"  Epoch {epoch+1:03d} | loss={train_loss:.3f} | val_loss={val_loss:.3f} | "
              f"lr={lr:.2e} | {elapsed:.0f}s {flag}")
 
        row = {
            "epoch"    : epoch + 1,
            "loss"     : round(train_loss, 6),
            "val_loss" : round(val_loss, 6),
            "lr"       : lr,
            "elapsed_s": round(elapsed, 1),
        }
        self.history_rows.append(row)
 
        # Simpan ke CSV setiap epoch (fault-tolerant)
        try:
            pd.DataFrame(self.history_rows).to_csv(self.csv_path, index=False)
        except Exception as e:
            print(f"    [WARNING] Gagal simpan CSV: {e}")
 
    def on_train_end(self, logs=None):
        best_epoch = min(self.history_rows, key=lambda r: r["val_loss"])
        print(f"Best: epoch {best_epoch['epoch']} | val_loss={best_epoch['val_loss']:.4f}")

KONFIGURASI

In [4]:
BASE_DIR       = os.path.join(PROJECT_ROOT, "Dataset_ocr")
PREP_DIR       = os.path.join(BASE_DIR, "preprocessed")
GT_CSV         = os.path.join(PREP_DIR, "ground_truth_auto.csv")
MODEL_DIR      = os.path.join(PROJECT_ROOT, "saved_model")
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoint")
LOG_DIR        = os.path.join(PROJECT_ROOT, "tensorboard_logs")
 
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
 
BATCH_SIZE    = 48
EPOCHS        = 30
LEARNING_RATE = 1e-4
PATIENCE      = 15 

LOAD DATASET

In [5]:
def load_dataset(split: str):
    arrays_dir = os.path.join(PREP_DIR, split, "arrays")
 
    if not os.path.exists(arrays_dir):
        print(f"  [ERROR] Folder tidak ada: {arrays_dir}")
        return None, None, None
 
    if not os.path.exists(GT_CSV):
        print(f"  [ERROR] ground_truth_auto.csv tidak ada: {GT_CSV}")
        return None, None, None
 
    df_gt = pd.read_csv(GT_CSV)
    df_gt = df_gt[df_gt["split"] == split].copy()
    df_gt = df_gt[df_gt["text"].notna() & (df_gt["text"].str.strip() != "")]
 
    fname_to_text = dict(zip(df_gt["filename"], df_gt["text"]))
    npy_count = len(glob.glob(os.path.join(arrays_dir, "*.npy")))
    print(f"  {split}: {len(df_gt)} ground truth | {npy_count} .npy")
 
    images, labels, classes = [], [], []
 
    for npy_path in sorted(glob.glob(os.path.join(arrays_dir, "*.npy"))):
        fname = Path(npy_path).name
        text  = fname_to_text.get(fname, None)
 
        if text is None or str(text).strip() == "":
            continue
 
        try:
            arr = np.load(npy_path, allow_pickle=False)
            if arr.shape != (IMG_H, IMG_W, CHANNELS):
                continue
 
            lbl = encode_text(str(text).lower()).numpy()
 
            if len(lbl) < MAX_TEXT_LENGTH:
                lbl = np.concatenate([
                    lbl,
                    np.full(
                        MAX_TEXT_LENGTH - len(lbl),
                        fill_value=PADDING_VALUE,
                        dtype=np.int32,
                    ),
                ])
            else:
                lbl = lbl[:MAX_TEXT_LENGTH]
 
            images.append(arr)
            labels.append(lbl.astype(np.int32))
            classes.append(fname)
 
        except Exception as e:
            print(f"  [SKIP] {fname}: {e}")
            continue
 
    if not images:
        print(f"  [WARNING] Tidak ada data untuk {split}")
        return None, None, None
 
    print(f"  Loaded: {len(images)} sampel")
    return (
        np.array(images, dtype=np.float32),
        np.array(labels, dtype=np.int32),
        classes,
    )

CUSTOM LOSS

In [6]:
def ctc_loss_fn(y_true, y_pred):
    """
    ✅ FIX Bug #1: blank_index=BLANK_INDEX=0 (konsisten dengan CTCLayer dan decode)
    ✅ FIX Bug #2: label_length dihitung berdasarkan PADDING_VALUE=-1, bukan NUM_CLASSES
 
    Sebelumnya: blank_index=NUM_CLASSES (86) → di luar range output (0-85)!
    """
    batch_len = tf.cast(tf.shape(y_true)[0], tf.int64)
 
    input_len = tf.cast(tf.shape(y_pred)[1], tf.int64) * \
                tf.ones(shape=(batch_len,), dtype=tf.int64)
 
    # Hitung panjang label asli (tanpa padding)
    label_len = tf.reduce_sum(
        tf.cast(tf.not_equal(y_true, PADDING_VALUE), tf.int64), axis=1
    )
 
    return tf.nn.ctc_loss(
        labels=tf.cast(y_true, tf.int32),
        logits=y_pred,
        label_length=tf.cast(label_len, tf.int32),
        logit_length=tf.cast(input_len, tf.int32),
        logits_time_major=False,
        blank_index=BLANK_INDEX    # ← BLANK_INDEX = 0
    )

HELPERS

In [7]:
idx_to_char = {i: c for i, c in enumerate(char_to_num.get_vocabulary())}
 
def edit_distance(s1: str, s2: str) -> int:
    """Levenshtein edit distance untuk CER."""
    m, n = len(s1), len(s2)
    dp   = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]

WEIGHTED SAMPLER

In [8]:
def make_weighted_dataset(X_train, y_train, batch_size):
    """
    Weighted sampling untuk atasi dominasi label pendek (QTY: "1","2","3").
    Sampel dengan teks lebih panjang diberi bobot lebih tinggi.
    """
    text_lengths = np.array([
        len([i for i in lbl if i > 0]) for lbl in y_train
    ], dtype=np.float32)
 
    # Bobot proporsional dengan panjang teks, min 1
    weights = np.clip(text_lengths, 1, None).astype(np.float64)
    weights = weights / weights.sum()
 
    n       = len(X_train)
    indices = np.random.choice(n, size=n, replace=True, p=weights)
    X_w, y_w = X_train[indices], y_train[indices]
 
    resampled_lengths = text_lengths[indices].astype(int)
    unique, counts = np.unique(resampled_lengths, return_counts=True)
    top5 = sorted(zip(unique, counts), key=lambda x: -x[1])[:5]
    print("  Distribusi panjang teks setelah resampling (top 5):")
    for length, count in top5:
        print(f"    {int(length):2d} karakter : {count} sampel")
 
    return tf.data.Dataset.from_tensor_slices(
        ({"image": X_w, "label": y_w}, y_w)
    ).shuffle(5000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

TRAINING LOOP


In [9]:
def train_with_gradient_tape(model, X_train, y_train, X_valid, y_valid, optimizer):
    
    train_ds = make_weighted_dataset(X_train, y_train, BATCH_SIZE)
 
    valid_ds = tf.data.Dataset.from_tensor_slices(
        ({"image": X_valid, "label": y_valid}, y_valid)
    ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
 
    writer    = tf.summary.create_file_writer(LOG_DIR)

    inf_model = build_inference_model(model)
 
    best_loss = np.inf
    wait      = 0
    history   = []
 
    current_lr  = LEARNING_RATE
    lr_patience = 5
    lr_wait     = 0
    lr_factor   = 0.5
    min_lr      = 1e-6
 
    print(f"\n[TRAINING] tf.GradientTape dimulai...")
    print(f"  Epochs     : {EPOCHS}")
    print(f"  Batch size : {BATCH_SIZE}")
    print(f"  Train      : {len(X_train)} sampel")
    print(f"  Valid      : {len(X_valid)} sampel")
    print(f"  LR awal    : {LEARNING_RATE}")
    print(f"  Patience   : {PATIENCE}\n")
 
    for epoch in range(EPOCHS):
 
        # ── Training ──────────────────────────────────────
        train_losses = []
        for batch_x, batch_y in train_ds:
            with tf.GradientTape() as tape:
                y_pred = model(batch_x, training=True)
                loss   = tf.reduce_mean(ctc_loss_fn(
                    tf.cast(batch_x["label"], tf.int32), y_pred
                ))
            grads = tape.gradient(loss, model.trainable_variables)
            # Clip gradients untuk stabilitas
            grads, _ = tf.clip_by_global_norm(grads, 5.0)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            train_losses.append(float(loss))
 
        # ── Validation ────────────────────────────────────
        val_losses = []
        for batch_x, batch_y in valid_ds:
            y_pred = model(batch_x, training=False)
            v_loss = tf.reduce_mean(ctc_loss_fn(
                tf.cast(batch_x["label"], tf.int32), y_pred
            ))
            val_losses.append(float(v_loss))
 
        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
 
 
        history.append({
            "epoch"    : epoch + 1,
            "loss"     : avg_train,
            "val_loss" : avg_val,
            "lr"       : current_lr,
        })
 
        with writer.as_default():
            tf.summary.scalar("loss",      avg_train,  step=epoch)
            tf.summary.scalar("val_loss",  avg_val,    step=epoch)
            tf.summary.scalar("lr",        current_lr, step=epoch)
 
        print(f"  Epoch {epoch+1:03d}/{EPOCHS} | loss={avg_train:.4f} | val_loss={avg_val:.4f} | "
              f"lr={current_lr:.2e} | wait={wait}/{PATIENCE}")
 
 
        # ── ReduceLROnPlateau ──────────────────────────────
        if avg_val < best_loss:
            best_loss = avg_val
            wait      = 0
            lr_wait   = 0
            best_path = os.path.join(CHECKPOINT_DIR, "best_model.keras")
            model.save(best_path)
            # Simpan ulang inference model
            inf_model = build_inference_model(model)
            print(f"    ✅ best_model.keras disimpan (val_loss={avg_val:.4f})")
        else:
            wait    += 1
            lr_wait += 1

            if lr_wait >= lr_patience and current_lr > min_lr:
                current_lr = max(current_lr * lr_factor, min_lr)
                optimizer.learning_rate.assign(current_lr)
                lr_wait = 0
                print(f"    📉 LR diturunkan → {current_lr:.2e}")
 
            if wait >= PATIENCE:
                print(f"    🛑 Early stopping di epoch {epoch+1}")
                break
 
    return history

MAIN

In [10]:
if __name__ == "__main__":
    print("[DATA] Memuat dataset dari ground_truth_auto.csv...")
    X_train, y_train, cls_train = load_dataset("train")
    X_valid, y_valid, cls_valid = load_dataset("valid")
    X_test,  y_test,  cls_test  = load_dataset("test")
 
    print()
    for split, X in [("train", X_train), ("valid", X_valid), ("test", X_test)]:
        if X is not None:
            print(f"  {split:5s}: {len(X):4d} sampel → shape={X.shape}")
        else:
            print(f"  {split:5s}: tidak ada data ❌")
 
    print(f"\n[MODEL] Membangun model...")
    model = build_ocr_model()
    print(f"  Params: {model.count_params():,}")
    model.summary()
 
    if X_train is not None and X_valid is not None:
        optimizer = keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=5.0
        )
 
        history = train_with_gradient_tape(
            model, X_train, y_train,
            X_valid, y_valid,
            optimizer=optimizer
        )
 
        pd.DataFrame(history).to_csv(
            os.path.join(PROJECT_ROOT, "training_history.csv"),
            index=False
        )
 
        config = {
            "model_name"   : "MoneyLens_OCR",
            "architecture" : "CNN(32-64-128) + BiLSTM(128-64) + CTC",
            "num_classes"  : NUM_CLASSES,
            "blank_index"  : BLANK_INDEX,
            "padding_value": PADDING_VALUE,
            "training": {
                "epochs_trained": len(history),
                "best_val_loss" : float(min(h["val_loss"] for h in history)),
                        "batch_size"    : BATCH_SIZE,
                "learning_rate" : LEARNING_RATE,
                "bugs_fixed"    : [
                    "blank_index=0 konsisten di seluruh pipeline",
                    "padding_value=-1 (bukan NUM_CLASSES)",
                    "softmax sebelum ctc_decode",
                    "build_inference_model robust via get_layer()",
                    "LR 1e-3 -> 1e-4 + ReduceLROnPlateau",
                ]
            },
            "data": {
                "train_samples": len(X_train),
                "valid_samples": len(X_valid),
                "test_samples" : len(X_test) if X_test is not None else 0,
                "ground_truth" : GT_CSV,
            }
        }
        with open(os.path.join(MODEL_DIR, "training_config.json"), "w") as f:
            json.dump(config, f, indent=2)
 
    else:
        print("\n[ERROR] Data tidak tersedia!")
        print(f"  Pastikan ground_truth_auto.csv ada di: {GT_CSV}")
 
    print(f"\n{'='*65}\nTRAINING SELESAI\n{'='*65}")
    print(f"  Checkpoint  : {CHECKPOINT_DIR}/best_model.keras")
    print(f"  TensorBoard : tensorboard --logdir={LOG_DIR}")
    print(f"  History     : {os.path.join(PROJECT_ROOT, 'training_history.csv')}")
    print(f"{'='*65}")

[DATA] Memuat dataset dari ground_truth_auto.csv...
  train: 3190 ground truth | 3194 .npy
  Loaded: 3172 sampel
  valid: 915 ground truth | 915 .npy
  Loaded: 915 sampel
  test: 422 ground truth | 422 .npy
  Loaded: 422 sampel

  train: 3172 sampel → shape=(3172, 32, 128, 1)
  valid:  915 sampel → shape=(915, 32, 128, 1)
  test :  422 sampel → shape=(422, 32, 128, 1)

[MODEL] Membangun model...
  Params: 3,967,830


Model: "MoneyLens_OCR_v2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 32, 128,   │          0 │ -                 │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_block_1         │ (None, 16, 64,    │     38,016 │ image[0][0]       │
│ (ResidualCNNBlock)  │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_block_2         │ (None, 8, 32,     │    230,400 │ cnn_block_1[0][0] │
│ (ResidualCNNBlock)  │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_block_3         │ (None, 4, 32,     │    919,552 │ cnn_block_2[0][0] │
│ (ResidualCNNBlock)  │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_block_4         │ (None, 4, 32,     │  1,181,696 │ cnn_block_3[0][0] │
│ (ResidualCNNBlock)  │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 32, 1024)  │          0 │ cnn_block_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_proj (Dense)  │ (None, 32, 128)   │    131,200 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 32, 128)   │          0 │ dense_proj[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 32, 512)   │    788,480 │ dropout_4[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 32, 256)   │    656,384 │ bilstm_1[0][0]    │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ label (InputLayer)  │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_logits (Dense) │ (None, 32, 86)    │     22,102 │ bilstm_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ctc_loss (CTCLayer) │ (None, 32, 86)    │          0 │ label[0][0],      │
│                     │                   │            │ char_logits[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,967,830 (15.14 MB)

 Trainable params: 3,965,014 (15.13 MB)

 Non-trainable params: 2,816 (11.00 KB)

  Distribusi panjang teks setelah resampling (top 5):
     5 karakter : 343 sampel
     4 karakter : 339 sampel
    32 karakter : 288 sampel
     6 karakter : 248 sampel
    10 karakter : 246 sampel

[TRAINING] tf.GradientTape dimulai...
  Epochs     : 30
  Batch size : 48
  Train      : 3172 sampel
  Valid      : 915 sampel
  LR awal    : 0.0001
  Patience   : 15



I0000 00:00:1779797319.998710  166145 cuda_dnn.cc:461] Loaded cuDNN version 92200
W0000 00:00:1779797321.211643  166145 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.57GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1779797323.337052  166145 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.64GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1779797325.214461  166145 bfc_allocator.cc:311] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.45GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1779797325.429459  166145 bfc_allocator.cc:311] 

  Epoch 001/30 | loss=101.0893 | val_loss=54.9317 | lr=1.00e-04 | wait=0/15
    ✅ best_model.keras disimpan (val_loss=54.9317)
  Epoch 002/30 | loss=85.6086 | val_loss=55.1025 | lr=1.00e-04 | wait=0/15
  Epoch 003/30 | loss=83.5616 | val_loss=55.5440 | lr=1.00e-04 | wait=1/15
  Epoch 004/30 | loss=83.0641 | val_loss=55.9091 | lr=1.00e-04 | wait=2/15
  Epoch 005/30 | loss=82.5524 | val_loss=54.1799 | lr=1.00e-04 | wait=3/15
    ✅ best_model.keras disimpan (val_loss=54.1799)
  Epoch 006/30 | loss=81.9562 | val_loss=51.2271 | lr=1.00e-04 | wait=0/15
    ✅ best_model.keras disimpan (val_loss=51.2271)
  Epoch 007/30 | loss=81.9354 | val_loss=48.3348 | lr=1.00e-04 | wait=0/15
    ✅ best_model.keras disimpan (val_loss=48.3348)
  Epoch 008/30 | loss=81.8165 | val_loss=48.0085 | lr=1.00e-04 | wait=0/15
    ✅ best_model.keras disimpan (val_loss=48.0085)
  Epoch 009/30 | loss=81.1590 | val_loss=47.1313 | lr=1.00e-04 | wait=0/15
    ✅ best_model.keras disimpan (val_loss=47.1313)
  Epoch 010/30 | l